# Train YOLOv8 — BISINDO ROI Detector (Google Colab)

Notebook untuk melatih YOLOv8 sebagai detektor `hand` / `person` (Region of Interest) untuk sistem SiBindo Translator.

**Pipeline:**
1. Setup environment (GPU + Ultralytics)
2. Mount Google Drive (untuk simpan dataset + weights)
3. Download dataset Roboflow / Kaggle
4. Train YOLOv8
5. Validasi (mAP)
6. Export weights `best.pt` ke Drive

**Saran**: pakai runtime **T4 GPU** (Colab Free) atau **A100** (Colab Pro). Aktifkan via *Runtime → Change runtime type → GPU*.

## 1. Setup Environment

In [1]:
# Cek GPU. Jika 'command not found' atau kosong: pilih runtime GPU di Colab.
!nvidia-smi || echo 'No GPU detected — training akan jalan di CPU (lambat).'

Tue May  5 13:58:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install Ultralytics + Roboflow client. Versi disamakan dengan requirements.txt repo.
!pip install -q ultralytics==8.4.46 roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 143.2 MB/s eta 0:00:0000:01


## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Konfigurasi

In [4]:
import os
import torch

PROJECT_DIR = '/content/drive/MyDrive/sibindo'
DATASET_DIR = f'{PROJECT_DIR}/datasets/yolo_bisindo'
RUNS_DIR    = f'{PROJECT_DIR}/runs/yolo'
WEIGHTS_OUT = f'{PROJECT_DIR}/weights/yolov8_best.pt'

EPOCHS     = 100
IMG_SIZE   = 640
BATCH      = 16
MODEL_BASE = 'yolov8n.pt'   # n / s / m / l / x
DEVICE     = 0 if torch.cuda.is_available() else 'cpu'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/weights', exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print('device =', DEVICE)

device = 0


## 4. Download Dataset (Roboflow)

Ganti `API_KEY`, workspace, project, dan version dengan project Roboflow Anda. Atau upload manual ke Drive.

In [5]:
from roboflow import Roboflow

API_KEY     = 'MBDa3NBbUecJlc9bdsHF'
WORKSPACE   = 'suhardis-workspace'
PROJECT     = 'hand-detection-2r6df-hqcwv'
VERSION_NUM = None   # set angka untuk pin versi, None = auto-pick terbaru

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)

versions = project.versions()
assert versions, 'Belum ada version di Roboflow. Buka web UI -> tab Versions -> Generate New Version.'
print('available versions:', [v.version for v in versions])

if VERSION_NUM is None:
    version = versions[-1]
    print('using latest version:', version.version)
else:
    version = project.version(VERSION_NUM)
    print('using pinned version:', VERSION_NUM)

dataset = version.download('yolov8', location=DATASET_DIR)
print('downloaded to:', dataset.location)

import os
print('Dataset path:', DATASET_DIR)
assert os.path.exists(f'{DATASET_DIR}/data.yaml'), \
    f'data.yaml tidak ditemukan di {DATASET_DIR}'

loading Roboflow workspace...
loading Roboflow project...
available versions: ['1']
using latest version: 1
downloaded to: /content/drive/MyDrive/sibindo/datasets/yolo_bisindo
Dataset path: /content/drive/MyDrive/sibindo/datasets/yolo_bisindo


In [6]:
!cat $DATASET_DIR/data.yaml

names:
- hand
nc: 1
roboflow:
  license: MIT
  project: hand-detection-2r6df-hqcwv
  url: https://universe.roboflow.com/suhardis-workspace/hand-detection-2r6df-hqcwv/dataset/1
  version: 1
  workspace: suhardis-workspace
test: ../test/images
train: ../train/images
val: ../valid/images


## 5. Training

In [7]:
from ultralytics import YOLO

model = YOLO(MODEL_BASE)
results = model.train(
    data=f'{DATASET_DIR}/data.yaml',
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name='bisindo_roi',
    project=RUNS_DIR,
    patience=20,
    save=True,
    plots=True,
    device=DEVICE,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/sibindo/datasets/yolo_bisindo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h

## 6. Validasi (mAP)

In [8]:
metrics = model.val()
print(f'mAP@0.5      = {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 = {metrics.box.map:.4f}')
print(f'precision    = {metrics.box.mp:.4f}')
print(f'recall       = {metrics.box.mr:.4f}')

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 20.3±12.9 MB/s, size: 23.7 KB)
val: Scanning /content/drive/MyDrive/sibindo/datasets/yolo_bisindo/valid/labels.cache... 182 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 182/182 42.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.5it/s 3.4s0.2s
                   all        182        280      0.864      0.815      0.886      0.488
Speed: 2.8ms preprocess, 6.4ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /content/runs/detect/val
mAP@0.5      = 0.8861
mAP@0.5:0.95 = 0.4879
precision    = 0.8638
recall       = 0.8153


## 7. Export Best Weights

In [9]:
import shutil
best = f'{RUNS_DIR}/bisindo_roi/weights/best.pt'
assert os.path.exists(best), f'best.pt tidak ditemukan di {best}'
shutil.copy(best, WEIGHTS_OUT)
print('Saved to', WEIGHTS_OUT)

Saved to /content/drive/MyDrive/sibindo/weights/yolov8_best.pt


## 8. Sample Inference

In [10]:
import glob
import cv2
from PIL import Image
from IPython.display import display

samples = sorted(glob.glob(f'{DATASET_DIR}/images/test/*.jpg'))[:3]
if not samples:
    print('Tidak ada sample di images/test/. Lewati.')
for s in samples:
    res = model.predict(s, save=False, verbose=False)[0]
    rgb = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    display(Image.fromarray(rgb))

Tidak ada sample di images/test/. Lewati.
